In [1]:
from matplotlib import pyplot as plt
from matplotlib import colors
import numpy as np
from matplotlib.patches import Path, PathPatch
import pandas as pd
from shapely.geometry import Point, shape, Polygon
from shapely.ops import unary_union, cascaded_union
from geopandas.tools import sjoin
import geopandas as gpd
from netCDF4 import Dataset
from cartopy import crs as ccrs
from cartopy.io.shapereader import Reader
from sklearn.metrics import mean_squared_error
import scipy.stats as st
from sklearn.linear_model import LinearRegression
import cartopy.feature as cfeature
import csv  
from matplotlib.colors import ListedColormap
from cartopy.feature import LAKES

C:\Users\x12la\AppData\Local\Temp\ipykernel_11856\2511498831.py:8: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  from geopandas.tools import sjoin


In [ ]:
# Load County boundary files
county = gpd.read_file('../US_county_2021.shp')
county = county.to_crs('EPSG:4326')

county_il =  county.loc[(county['STATEFP']=='17')]
county_il["STATEFP"] = county_il["STATEFP"].astype(str).str.zfill(2)
county_il["COUNTYFP"] = county_il["COUNTYFP"].astype(str).str.zfill(3)
county_il["region_cd"] = county_il["STATEFP"] + county_il["COUNTYFP"]

In [ ]:
# Gridded ~!km LOCUS data for Illinois 
base = gpd.read_file('LOCUS_IL_Idling.shp')
base = base.to_crs('EPSG:4326')

In [ ]:
# make copies 
county_proj = county_il.to_crs(3857)
base_proj = base.to_crs(3857)

# grid area based on geom
base_proj["grid_area"] = base_proj.geometry.area

# find grid intersecting county boundaries
intersections = gpd.overlay(base_proj, county_proj, how="intersection")

# calc how much area is intersected with county 
# this is used to calculate area weighted of gridded idling hours
# to used a weighted area average to determine county idling totals 
# this is done by calc frac of area intersecting the county
# then multipliying the frac to the gridded today hours
intersections["intersect_area"] = intersections.geometry.area
intersections["frac"] = intersections["intersect_area"] / intersections["grid_area"]

for col in ["MDT_avg_da", "HDT_avg_da", "Total_avg_"]:
    intersections[f"{col}_weighted"] = intersections[col] * intersections["frac"]

# generate county totals (sum over all grid cells within and intersecting county boundary) 
county_activity = (
    intersections.groupby("GEOID")[[
        "MDT_avg_da_weighted", 
        "HDT_avg_da_weighted", 
        "Total_avg__weighted",
    ]]
    .sum()
    .reset_index()
)

# merge back to shapefile 
county_out = county_il.merge(county_activity, on="GEOID", how="left")

In [ ]:
# county_out.to_file('LOCUS_CountyIdling.shp')
county_out.head()